In [0]:
silver_table=dbutils.jobs.taskValues.get("BronzeToSilver","silver_table",debugValue="onlineretail_ws.silver.online_retail")
dbutils.widgets.text('gold_dim_table','onlineretail_ws.gold.dim_customer')
dim_table=dbutils.widgets.get('gold_dim_table')
dbutils.widgets.text("gold_dimension","abfss://gold@onlineretailproject.dfs.core.windows.net/dimensions/")
gold_dimension=dbutils.widgets.get("gold_dimension")
dbutils.jobs.taskValues.set('dim_table',dim_table)

In [0]:
gold_dim=spark.sql(f"SELECT Country,Invoice,StockCode,Quantity,Price,InvoiceDate,Customer_ID,currency_code,Is_Cancelled FROM {silver_table} WHERE is_active='Y'")

In [0]:
gold_dim=spark.sql(f"OPTIMIZE {dim_table} ZORDER BY (InvoiceDate)")

In [0]:
gold_dim.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .option("path", gold_dimension) \
    .saveAsTable(dim_table)